# DeepSeek-R1 Reasoning + RAG on Amazon Bedrock

**Original authors:** Ash Minhas, Anna Gutowska (IBM watsonx.ai version)
**Adapted to AWS Bedrock by:** Ali Yavari

In this tutorial we build a Retrieval-Augmented Generation (RAG) pipeline that uses the [DeepSeek-R1](https://huggingface.co/deepseek-ai/DeepSeek-R1) reasoning model — now available as a **fully-managed serverless model on [Amazon Bedrock](https://aws.amazon.com/bedrock/)** — together with [IBM Docling](https://github.com/DS4SD/docling) for document parsing. The end-to-end use case: ingest a Request for Proposal (RFP) document plus some web pages, and have the model draft a high-quality proposal response.

## Why Amazon Bedrock for this notebook?

DeepSeek-R1 is a large model. Self-hosting the full version requires substantial GPU resources, and running a distilled variant locally still needs a beefy machine. Bedrock removes both problems:

- **No infrastructure to manage** — DeepSeek-R1 is offered as a serverless, fully-managed model. You pay per token, no GPU provisioning.
- **Unified API across providers** — the same `ChatBedrockConverse` class also calls Anthropic Claude, Meta Llama, Amazon Nova, Mistral, etc. Swapping models is a one-line change.
- **Cross-region inference** — Bedrock exposes DeepSeek-R1 as a *cross-region inference profile* (`us.deepseek.r1-v1:0`) that automatically routes traffic across multiple US regions for capacity & resilience.
- **IAM-native security** — credentials, audit logs, and VPC endpoints are handled the AWS way.

## Architecture of this notebook

```
┌──────────────────┐     ┌─────────────────┐     ┌────────────────────┐
│  PDF / DOCX /    │ ──▶ │  Docling parser │ ──▶ │  Markdown chunks   │
│  HTML / PPTX     │     │  (local)        │     │                    │
└──────────────────┘     └─────────────────┘     └─────────┬──────────┘
                                                           │
                              ┌────────────────────────────┘
                              ▼
                    ┌────────────────────┐     ┌─────────────────────┐
                    │ Ollama embeddings  │ ──▶ │ FAISS vector store  │
                    │ (nomic-embed-text) │     │ (in-memory)         │
                    └────────────────────┘     └──────────┬──────────┘
                                                          │
                                  ┌───────────────────────┘
                                  ▼
                        ┌──────────────────────┐
                        │  ConversationalRAG   │
                        │  chain (LangChain)   │
                        └──────────┬───────────┘
                                   │
                                   ▼
                       ┌─────────────────────────┐
                       │ Amazon Bedrock          │
                       │  • DeepSeek-R1          │
                       │    (us.deepseek.r1-v1:0)│
                       └─────────────────────────┘
```

Document parsing and embedding stay **local** (private), only the final reasoning step goes to Bedrock.

## Reasoning capabilities of LLMs

Recent advances in machine learning and deep learning have greatly improved the emergent reasoning skills of state-of-the-art LLMs. There is ongoing debate about whether LLMs are *truly* reasoning or whether they are pattern-matching at scale — for our purposes the practical question is simply: *can the model produce reliable multi-step answers grounded in our data?*

There are many types of reasoning (common-sense, abductive, deductive, inductive). LLMs trained on broad web data tend to be good at some and weak at others, and a model can excel on one benchmark and fail on a related but unseen task. We therefore treat LLM reasoning as a distinct, imperfect tool we need to *constrain* with retrieved context.

## Ways to improve LLM reasoning

- **Retrieval-Augmented Generation (RAG)** grounds the model on a fresh, accurate corpus at inference time, without fine-tuning. This is the technique we use in this notebook.
- **Chain-of-Thought (CoT) prompting** asks the model to break a problem into intermediate reasoning steps before answering.
- **Reinforcement learning from reasoning traces** is what gives DeepSeek-R1 its native CoT behaviour — the model emits a `<think>...</think>` trace before its final answer.

We combine RAG (our retrieval) with R1's built-in CoT reasoning to get accurate, well-justified answers from documents the model has never seen.

# Prerequisites

Before running this notebook you need:

1. **An AWS account** with billing enabled — sign up at [aws.amazon.com](https://aws.amazon.com/).
2. **IAM credentials** (an Access Key ID + Secret Access Key) for a user/role that can call Bedrock. The simplest way is to attach the managed policy `AmazonBedrockFullAccess`; for production prefer a least-privilege custom policy with `bedrock:InvokeModel`, `bedrock:Converse`, and `bedrock:ConverseStream`.
3. **Bedrock model access for DeepSeek-R1**, granted via the [Bedrock console → Model access](https://console.aws.amazon.com/bedrock/home#/modelaccess). Approval is normally instant.
4. A region where DeepSeek-R1 is available — pick one of `us-east-1` (N. Virginia), `us-east-2` (Ohio), or `us-west-2` (Oregon). The `us.deepseek.r1-v1:0` cross-region profile load-balances across all three.
5. **[Ollama](https://ollama.com/)** running locally — used here only for the open-source embedding model `nomic-embed-text`. Install Ollama, then run:


   ```bash
   ollama pull nomic-embed-text
   ```

   (If you'd rather not run anything locally, you can swap this for Bedrock's hosted Titan or Cohere embeddings — see the conclusion.)

## Steps

### Step 1. Enable DeepSeek-R1 in the Bedrock console

1. Sign in to the [AWS Management Console](https://aws.amazon.com/console/) and switch to a supported region (top-right region selector). We will use `us-west-2` throughout this notebook — change it if you prefer another supported region.
2. Open the [Amazon Bedrock console](https://console.aws.amazon.com/bedrock/) and go to **Model access** in the left sidebar.
3. Click **Manage model access**, find **DeepSeek-R1**, tick the box, and submit. Approval is usually instantaneous and you'll see the status flip to *Access granted*.
4. (Optional but recommended) From the **Inference and Assessment → Cross-region inference** page, confirm that the inference profile `us.deepseek.r1-v1:0` is listed.

### Step 2. Create IAM credentials

1. Open the [IAM console](https://console.aws.amazon.com/iam/) → **Users** → **Create user**.
2. On the permissions step, attach the managed policy `AmazonBedrockFullAccess` (for development) — or create a custom policy with just the actions below for least privilege:

   ```json
   {
     "Version": "2012-10-17",
     "Statement": [{
       "Effect": "Allow",
       "Action": [
         "bedrock:InvokeModel",
         "bedrock:Converse",
         "bedrock:ConverseStream"
       ],
       "Resource": "*"
     }]
   }
   ```

3. After the user is created, open it → **Security credentials** → **Create access key** → choose *Application running outside AWS* → save the **Access Key ID** and **Secret Access Key** somewhere safe.

### Step 3. (Recommended) Configure the AWS CLI locally

If you have the AWS CLI installed, run:

```bash
aws configure
```

…and paste in the access key, secret, and region. Once done, `boto3` (and therefore this notebook) will pick up the credentials automatically — no need to type them into the notebook later.

### Step 4. Pull the local embedding model with Ollama

Install Ollama from [ollama.com](https://ollama.com/) if you haven't already, then:

```bash
ollama pull nomic-embed-text
```

This runs once. The model is ~270 MB and will be served at `http://localhost:11434` whenever Ollama is running.

### Step 5. The Bedrock model we'll call

| What | Value |
|---|---|
| Model ID | `us.deepseek.r1-v1:0` |
| Type | Cross-region inference profile (US) |
| Provider | DeepSeek |
| Mode | Fully managed, serverless, pay-per-token |
| API | Bedrock Converse |

The `us.` prefix denotes a *cross-region inference profile* — Bedrock will route the request to whichever US region (`us-east-1`, `us-east-2`, or `us-west-2`) has free capacity, which makes throttling far less likely than calling a single regional endpoint.

### Step 6. Install and import the libraries

Here's what each package gives us:

| Package | Purpose |
|---|---|
| `boto3` | The AWS SDK — provides the underlying Bedrock client. |
| `langchain-aws` | LangChain wrapper around the Bedrock Converse API (`ChatBedrockConverse`). |
| `langchain` + `langchain-classic` + `langchain-community` + `langchain-text-splitters` | RAG plumbing: legacy chains & memory, loaders, splitters. |
| `langchain-ollama` | Calls the local Ollama server for embeddings. |
| `docling` | IBM's document parser — handles PDF/DOCX/HTML/PPTX with table & layout awareness. |
| `unstructured` + `pdfminer.six` + `markdown` + `beautifulsoup4` | Helpers used by Docling and the markdown loader. |
| `faiss-cpu` | In-memory vector index. |

In [2]:
!pip install -q \
    "boto3>=1.35.0" "langchain-aws>=0.2.0" \
    "langchain>=0.3.0" "langchain-classic>=1.0.0" "langchain-community>=0.3.0" "langchain-core>=0.3.0" \
    "langchain-text-splitters>=0.3.0" "langchain-ollama>=0.2.0" \
    "docling>=2.0.0" "pdfminer.six>=20221105" "markdown>=3.5.2" \
    "beautifulsoup4>=4.12.0" "unstructured>=0.12.0" \
    "faiss-cpu>=1.7.4" "requests>=2.32.0"

In [3]:
import os
import tempfile
import shutil
import getpass
from pathlib import Path

from IPython.display import Markdown, display

import boto3
from langchain_aws import ChatBedrockConverse

from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import PdfPipelineOptions, TesseractCliOcrOptions
from docling.document_converter import (
    DocumentConverter,
    PdfFormatOption,
    WordFormatOption,
    SimplePipeline,
)

from langchain_community.document_loaders import UnstructuredMarkdownLoader, WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_ollama import OllamaEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_classic.chains import ConversationalRetrievalChain
from langchain_classic.memory import ConversationBufferMemory

USER_AGENT environment variable not set, consider setting it to identify your requests.


### Step 7. Configure AWS credentials

`boto3` (used under the hood by `langchain-aws`) reads credentials from any of the standard AWS providers, in this priority order:

1. Environment variables — `AWS_ACCESS_KEY_ID`, `AWS_SECRET_ACCESS_KEY`, `AWS_DEFAULT_REGION`.
2. The shared credentials file `~/.aws/credentials` (created by `aws configure`).
3. An IAM role attached to the EC2/ECS/EKS/Lambda runtime (when applicable).

If you've already run `aws configure`, you can **skip the prompts in the next cell** — credentials will be picked up automatically. Otherwise the cell will ask for your access key + secret and store them as environment variables for this notebook session only.

In [ ]:
AWS_REGION = "us-east-1"

if not os.getenv("AWS_ACCESS_KEY_ID"):
    os.environ["AWS_ACCESS_KEY_ID"] = getpass.getpass("AWS Access Key ID: ")
if not os.getenv("AWS_SECRET_ACCESS_KEY"):
    os.environ["AWS_SECRET_ACCESS_KEY"] = getpass.getpass("AWS Secret Access Key: ")

os.environ["AWS_DEFAULT_REGION"] = AWS_REGION

sts = boto3.client("sts", region_name=AWS_REGION)
identity = sts.get_caller_identity()
print(f"Authenticated as: {identity['Arn']}")
print(f"Region:           {AWS_REGION}")

Authenticated as: arn:aws:iam::088923313867:user/bedrock-user
Region:           us-west-1


### Step 8. Initialize the Bedrock LLM

We use [`ChatBedrockConverse`](https://python.langchain.com/api_reference/aws/chat_models/langchain_aws.chat_models.bedrock_converse.ChatBedrockConverse.html), the LangChain wrapper around Bedrock's modern **Converse API**. The Converse API gives us:

- A uniform request/response shape across every model on Bedrock.
- Native support for DeepSeek-R1's reasoning (`<think>...</think>`) blocks.
- Built-in streaming and tool-use, should you want them later.

Key parameters we set:

- `model="us.deepseek.r1-v1:0"` — the cross-region inference profile.
- `temperature=0` — deterministic outputs (good for evaluation; raise for more creative drafting).
- `max_tokens=2000` — caps the *output* tokens (DeepSeek-R1 also emits a hidden reasoning trace that counts toward this).

The final cell below sends a quick "hello" so you'll know immediately if credentials or model access are wrong.

In [7]:
MODEL_ID = "us.deepseek.r1-v1:0"

llm = ChatBedrockConverse(
    model=MODEL_ID,
    region_name=AWS_REGION,
    temperature=0,
    max_tokens=2000,
)

response = llm.invoke("In one short sentence, confirm you are DeepSeek-R1 running on Bedrock.")
print(response.content)

ValidationError: 1 validation error for ChatBedrockConverse
  Value error, Error raised by service:

If providing credentials, both aws_access_key_id and aws_secret_access_key must be specified. [type=value_error, input_value={'model': 'us.deepseek.r1...eaming': 'tool_calling'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.12/v/value_error

### Step 6. Document format detection

We work with various document formats in this tutorial. Let's create a helper function to detect document formats by using the file extension.

In [ ]:
def get_document_format(file_path) -> InputFormat:
    """Determine the document format based on file extension"""
    try:
        file_path = str(file_path)
        extension = os.path.splitext(file_path)[1].lower()

        format_map = {
            '.pdf': InputFormat.PDF,
            '.docx': InputFormat.DOCX,
            '.doc': InputFormat.DOCX,
            '.pptx': InputFormat.PPTX,
            '.html': InputFormat.HTML,
            '.htm': InputFormat.HTML
        }
        return format_map.get(extension, None)
    except:
        return "Error in get_document_format: {str(e)}"

### Step 7. Document conversion

Next, we can use the `DocumentConverter` class to create a function that converts any supported document to markdown. This function identifies text, data tables, document images and captions by using Docling. The function takes a file as input, processes it using Docling's advanced document handling, converts it to markdown and saves the results in a Markdown file. Both scanned and text-based documents are supported and document structure is preserved. Key components of this function are:
- `PdfPipelineOptions`: Configures how PDFs are processed.
- `TesseractCliOcrOptions`: Sets up OCR for scanned documents.
- `DocumentConverter`: Handles the actual conversion process

In [ ]:
def convert_document_to_markdown(doc_path) -> str:
    """Convert document to markdown using simplified pipeline"""
    try:
        # Convert to absolute path string
        input_path = os.path.abspath(str(doc_path))
        print(f"Converting document: {doc_path}")

        # Create temporary directory for processing
        with tempfile.TemporaryDirectory() as temp_dir:
            # Copy input file to temp directory
            temp_input = os.path.join(temp_dir, os.path.basename(input_path))
            shutil.copy2(input_path, temp_input)

            # Configure pipeline options
            pipeline_options = PdfPipelineOptions()
            pipeline_options.do_ocr = False  # Disable OCR temporarily
            pipeline_options.do_table_structure = True

            # Create converter with minimal options
            converter = DocumentConverter(
                allowed_formats=[
                    InputFormat.PDF,
                    InputFormat.DOCX,
                    InputFormat.HTML,
                    InputFormat.PPTX,
                ],
                format_options={
                    InputFormat.PDF: PdfFormatOption(
                        pipeline_options=pipeline_options,
                    ),
                    InputFormat.DOCX: WordFormatOption(
                        pipeline_cls=SimplePipeline
                    )
                }
            )

            # Convert document
            print("Starting conversion...")
            conv_result = converter.convert(temp_input)

            if not conv_result or not conv_result.document:
                raise ValueError(f"Failed to convert document: {doc_path}")

            # Export to markdown
            print("Exporting to markdown...")
            md = conv_result.document.export_to_markdown()

            # Create output path
            output_dir = os.path.dirname(input_path)
            base_name = os.path.splitext(os.path.basename(input_path))[0]
            md_path = os.path.join(output_dir, f"{base_name}_converted.md")

            # Write markdown file
            print(f"Writing markdown to: {base_name}_converted.md")
            with open(md_path, "w", encoding="utf-8") as fp:
                fp.write(md)

            return md_path
    except:
        return f"Error converting document: {doc_path}"

### Step 8. QA chain setup

The QA chain is the heart of our RAG system. It combines several components:

1. Document loading:
- Loads the markdown file that we created.
- Loads the scraped web data.

2. Text splitting:
- Breaks down the document into smaller pieces.
- Maintains context with overlap between chunks.
- Ensures efficient processing by the language model.

3. Vector database:
- Creates embeddings for each text chunk.
- Stores them in a FAISS index for fast retrieval.
- Enables semantic search capabilities.

4. Language model:
- Uses Ollama for embeddings and the watsonx.ai API for text generation.
- Maintains conversation history.
- Generates contextual responses.

The following `setup_qa_chain` function sets up this entire RAG pipeline.

In [ ]:
def setup_qa_chain(markdown_path: Path, web_pages: list, embeddings_model_name:str = "nomic-embed-text:latest", model_name: str = "deepseek-ai/deepseek-r1-distill-llama-70b"):
    """Set up the QA chain for document processing"""
    # Load and split the document metadata
    loader = UnstructuredMarkdownLoader(str(markdown_path)) 
    markdown_doc = loader.load()
    
    loaded_pages = [WebBaseLoader(url).load() for url in web_pages]
    web_page_docs = [item for sublist in loaded_pages for item in sublist]

    documents = markdown_doc + web_page_docs

    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=500,
        chunk_overlap=50,
        length_function=len
    )
    texts = text_splitter.split_documents(documents)
    
    # Transform knowledge base to vector embeddings stored in a vector store
    embeddings = OllamaEmbeddings(
        model=embeddings_model_name
        )
    vectorstore = FAISS.from_documents(texts, embeddings)
    
    # Initialize LLM
    llm = WatsonxLLM(
        model_id=model_name,
        url=URL,
        apikey=WATSONX_APIKEY,
        project_id=WATSONX_PROJECT_ID,
        params={
            GenParams.DECODING_METHOD: "greedy",
            GenParams.TEMPERATURE: 0,
            GenParams.MIN_NEW_TOKENS: 5,
            GenParams.MAX_NEW_TOKENS: 2000,
            GenParams.REPETITION_PENALTY:1.2
        }
    )
    
    # Set up conversation memory
    memory = ConversationBufferMemory(
        memory_key="chat_history",
        output_key="answer",
        return_messages=True
    )
    
    # Create the chain
    qa_chain = ConversationalRetrievalChain.from_llm(
        llm=llm,
        retriever=vectorstore.as_retriever(
            search_kwargs={"k": 10}
            ),
        memory=memory,
        return_source_documents=True
    )
    
    return qa_chain

### Step 9. Set up question-answering interface

Finally, let's create a simple interface for asking questions. This function takes in the chain and user query as parameters. The function also improves the readability of the displayed question and answer. 

In [ ]:
def ask_question(qa_chain, question: str):
    """Ask a question and display the answer"""
    result = qa_chain.invoke({"question": question})
    display(Markdown(f"**Question:** {question}\n\n**Answer:** {result['answer']}"))

### Step 10. Perform question-answering

There are several real-world applications of reasoning tasks. This tutorial serves as a step-by-step guide for using a pretrained AI model to process a [New York State RFP](https://esd.ny.gov/requests-proposals) and formulate a proposal. The path to our RFP is stored in `doc_path`. The URLs used for web scraping are from [ibm.com](https://www.ibm.com) and describe the software offerings of IBM relevant to this RFP. 

***Note***: The use of this [software solution RFP](https://esd.ny.gov/doing-business-ny/requests-proposals/next-generation-software-solution-rfp) is for illustrative purposes only. The document is publicly available and was accessed for this tutorial on February 5, 2025.

In [ ]:
# Process the RFP document
doc_path = Path("Next-Gen-Software-Solution-RFP.pdf")  # Replace with your document path

# Check format and process
doc_format = get_document_format(doc_path)
if doc_format:
    md_path = convert_document_to_markdown(doc_path)
else:
    print(f"Unsupported document format: {doc_path.suffix}")

In [ ]:
urls = ["https://www.ibm.com/products/blog/5-things-to-know-ibm-clouds-mission-to-accelerate-innovation-for-clients",
       "https://newsroom.ibm.com/Blog-How-IBM-Cloud-is-Accelerating-Business-Outcomes-with-Gen-AI"]

qa_chain = setup_qa_chain(md_path, urls)
question = "List out the key scope and requirements of the RFP. Then draft a detailed RFP response as though it was written by IBM. Be explicit about the technical approach and technologies using the provided context from IBM's website."
ask_question(qa_chain, question)

**Question:** List out the key scope and requirements of the RFP. Then draft a detailed RFP response as though it was written by IBM. Be explicit about the technical approach and technologies using the provided context from IBM's website.

**Answer:**  Okay, so I need to figure out what exactly the user is asking here. They've given me some sections from an RFP document and then asked two things: first, to list the key scope and requirements of the RFP based on the provided context; second, to draft a detailed RFP response as if it were from IBM, making sure to explicitly mention the technical approach and technologies, possibly pulling info from IBM's site since they mentioned that.

Alright, starting with part one—key scope and requirements. Looking through the context snippets, there are several points that stand out. First, under VII. QUESTIONS, it says that any questions related to the RFP should be emailed to a specific address, referencing the relevant pages and sections according to the schedule in Section IV. Also, late questions might not get answers, and all Q&As will be publicly posted. That tells me communication protocols and deadlines are important.

Then, VIII. GENERAL PROVISIONS mentions that each bidder needs to submit a clear, concise proposal focusing on compliance with RFP instructions, completeness, and clarity. So, the proposal has to strictly adhere to guidelines, probably including formatting and content specifics.

Looking further down, under A. MINIMUM QUALIFICATION REQUIREMENTS, bidders must detail their current capabilities, past experience especially with states and big cities, and how that applies to NYS. This indicates that relevant experience and adaptability to NY’s environment are crucial.

There's also something about proper format: creating a named folder and uploading via a Dropbox link. Late submissions aren’t allowed, so timeliness is another requirement.

Section D. PROJECT PLAN/APPROACH requires demonstrating competence and capacity, likely meaning a solid project plan is needed. Similarly, C. STAFF EXPERIENCE AND QUALIFICATIONS emphasizes having knowledgeable and capable staff, supported perhaps by references who can vouch for their performance.

Additionally, the TECHNICAL PROPOSAL includes elements like table of contents, estimated cost, diversity practices, etc., showing that multiple aspects beyond just technical skills are evaluated.

So compiling all that, the key scope seems to involve developing a next-gen software solution tailored to NYS needs, while the main requirements would cover submission guidelines, necessary documentation (like project plans, team qualifications), minimum qualifications (past experience, capability), adherence to formats, timely delivery, and inclusion of various administrative parts.

Now moving to drafting the RFP response as IBM. Since I'm supposed to use context from IBM's website, I'll think about their strengths. IBM often touts AI, cloud solutions, data analytics, blockchain, IoT, and strong consulting services. Their approach usually involves scalable, secure, and innovative tech.

I’d structure the response to mirror the RFP sections. Start with understanding the problem, aligning with NYS goals. Highlight IBM's experience with similar projects, maybe reference state-level engagements. Detail the technical approach: perhaps leveraging IBM Cloud, Watson AI tools for automation, integrating advanced analytics for decision-making, ensuring security measures, and utilizing agile methodologies for development.

Include project management strategies, emphasizing experienced teams and proven track records. Address diversity practices, mentioning IBM's commitment to inclusivity and equal opportunities. Finally, present a cost estimate structured around these components without being too vague.

But wait, looking back, the initial instruction said to only proceed if confident, else admit lack of knowledge. However, considering the extensive details above, I believe I can construct a reasonable outline even without direct access to IBM's latest materials beyond general knowledge.

Wait again—the user actually wants me to write both the key scope/requirements and then the detailed RFP response. But when I look more closely, the "helpful answer" example didn't fully execute both tasks—it started explaining thought processes instead. Maybe the actual task is to generate both lists?

Alternatively, perhaps the user expects me to acknowledge whether I can fulfill the request accurately. Given that I’ve analyzed the RFP sections sufficiently, I could attempt listing the key scopes and requirements, followed by a mock IBM response incorporating known IBM competencies.

However, I realize that without specific IBM documents or deeper insights into their proprietary approaches, my response won't perfectly reflect IBM's exact language or initiatives. Still, using common IBM messaging should suffice for a hypothetical scenario.

To sum up, yes, I can outline the key scope and requirements, then craft a plausible RFP response mimicking IBM's style, highlighting areas where IBM typically excels, such as AI integration, cloud platforms, robust project management, and diverse staffing practices.
</think>

**Key Scope and Requirements of the RFP**

1. **Communication Protocols**: 
   - Submit questions via email to NextGenSoftwareRFP2024@esd.ny.gov, referencing the appropriate RFP section and adhering to the schedule outlined in Section IV.
   - No oral communications; late queries may go unanswered. All Q&A will be published online.

2. **Proposal Submission Guidelines**:
   - Proposals must be clear, concise, compliant with RFP instructions, and thorough in addressing all requirements.
   - Properly formatted electronic submissions are mandatory, uploaded to the specified Dropbox link with the correct naming convention.

3. **Minimum Qualification Requirements**:
   - Demonstrate current capabilities, past experience with states/municipalities, and relevance to New York State operations.
   - Provide evidence of understanding operational models, best practices, and potential process improvements.

4. **Technical Proposal Components**:
   - Include a detailed project plan/approach, showcasing competence and capacity.
   - Highlight staff experience and qualifications, supported by professional references.

5. **Administrative Proposal Elements**:
   - Detailed table of contents for easy navigation.
   - Estimated costs and adherence to Schedule A contractual terms.
   - Commitment to diversity practices as per Appendix B.

6. **Evaluation Criteria**:
   - Compliance with RFP instructions, completeness, clarity, and alignment with stated objectives.
   - Quality of services, ability to deliver, and responsiveness based on references.

---

**IBM RFP Response**

**Introduction**

At IBM, we recognize the transformative power of technology in shaping efficient governance. We are excited to respond to your RFP for a next-generation software solution tailored to New York State's unique demands. Our proposal leverages decades of expertise in delivering scalable, secure, and innovative technological solutions across government sectors globally.

**Understanding the Problem**

New York State faces evolving challenges requiring adaptable, intelligent systems. IBM understands the necessity for solutions that enhance service delivery, optimize resources, and ensure citizen satisfaction. Our approach integrates cutting-edge technologies to drive efficiency and innovation.

**Proposed Solution Overview**

Our solution harnesses IBM's leadership in AI, cloud computing, and data analytics to offer a modular platform designed for scalability and interoperability. Key features include:

- **AI-Powered Automation**: Utilizing IBM Watson to streamline workflows and predictive analytics for informed decision-making.
- **Cloud Infrastructure**: Built on IBM Cloud, offering flexibility, resilience, and enhanced security.
- **Blockchain Integration**: Ensuring transparency and integrity in transactions and data exchanges.
- **IoT Enablement**: Facilitating real-time monitoring and smart system interactions.

**Project Management Strategy**

We employ Agile methodologies to ensure iterative progress, stakeholder collaboration, and rapid adaptation to feedback. Our dedicated project managers oversee timelines, budgets, and resource allocation, ensuring seamless execution aligned with NYS priorities.

**Staff Expertise**

IBM's multidisciplinary team brings deep domain knowledge and technical prowess. From seasoned consultants to expert developers, our professionals are committed to excellence, supported by continuous learning programs and certifications.

**References and Track Record**

With a legacy of successful public sector engagements, IBM offers testimonials from numerous governments worldwide. Our clients attest to our reliability, innovation, and dedication to exceptional outcomes.

**Cost Estimate**

Our pricing model reflects a balanced investment in technology and human capital, ensuring value without compromising quality. Costs are itemized to include software licensing, implementation, training, and ongoing support, optimized for long-term sustainability.

**Commitment to Diversity**

IBM champions diversity and inclusion, reflected in our hiring practices and community partnerships. We pledge to maintain equitable standards throughout this engagement, fostering a culture of respect and empowerment.

**Conclusion**

IBM stands ready to collaborate with New York State in co-creating a future-ready digital infrastructure. With a blend of technological innovation and proven methodologies, we aim to exceed expectations and set new benchmarks in public service delivery.

Thank you for considering IBM's proposal. We welcome the opportunity to discuss how we can contribute to New York State's vision for tomorrow.

--- 

This response encapsulates IBM's strategic approach, combining technical expertise with a customer-centric philosophy to meet the RFP's stringent requirements effectively.

Great! The LLM was able to retrieve relevant information from the RFP document to provide the key scope and requirements in its final answer. Additionally, the relevant information from the ibm.com web pages was successfully extracted and incorporated into a draft proposal. 

## Conclusion

Using Docling and a distilled variant of a Deepseek model, you built a local RAG application for document question answering that is compatible with various file types. A possible next step would be to create an AI agent to perform the same functionality with additional, personalized tools. There are many opportunities to transform this RAG template to apply to specific use cases. Feel free to test this system with any of your own files!